In [1]:
from calibratedRecs.calibration import Calibration

In [2]:
from dynamicTasteDistortion.dataset_loader import get_ml_df

data_type = "ml"
size = "s"

raw_df = get_ml_df(size)

Skipping: 'dynamicTasteDistortion/data/movielens/raw/ml_s/' already exists and contains files.
Joining movies data with ratings data...
Preprocessing dataset...


## Build item genre distribution tensor

In [3]:
from calibratedRecs.constants import ITEM_COL, GENRE_COL
from calibratedRecs.calibrationUtils import normalize_counter, merge_dicts, preprocess_dataframe_for_calibration
from typing import Counter

import torch

In [4]:
df = preprocess_dataframe_for_calibration(raw_df)

In [5]:
df.rating.min()

2.1202273e-05

In [6]:
df.rating.max()

0.16488238

In [8]:
item2genre = df[[ITEM_COL, GENRE_COL]].drop_duplicates()
all_genres = item2genre.explode("genres")["genres"].drop_duplicates().tolist()
n_genres = len(all_genres)
std_dict = {genre: 0 for genre in all_genres}

In [9]:

item2genre["genre_dist"] = item2genre["genres"].apply(
    lambda genre_list: normalize_counter(Counter(genre_list))
)

In [10]:
item2genre

,item,genres,genre_dist
0,1,"(animation, children's, comedy)","{'animation': 0.3333333333333333, 'children's'..."
2077,2,"(adventure, children's, fantasy)","{'adventure': 0.3333333333333333, 'children's'..."
2778,3,"(comedy, romance)","{'comedy': 0.5, 'romance': 0.5}"
3256,4,"(comedy, drama)","{'comedy': 0.5, 'drama': 0.5}"
3426,5,"(comedy,)",{'comedy': 1.0}
...,...,...,...
998561,3948,"(comedy,)",{'comedy': 1.0}
999423,3949,"(drama,)",{'drama': 1.0}
999727,3950,"(drama,)",{'drama': 1.0}
999781,3951,"(drama,)",{'drama': 1.0}


In [11]:
n_items = item2genre.item.max() + 1
genre_vector = (
    item2genre["genre_dist"]
    .apply(lambda count: merge_dicts(std_dict, count))
    .apply(lambda dictionary: dict(sorted(dictionary.items())))
    .apply(lambda dictionary: list(dictionary.values()))
).tolist()

In [12]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
genre_vector

[[0,
  0,
  0.3333333333333333,
  0.3333333333333333,
  0.3333333333333333,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [0,
  0.3333333333333333,
  0,
  0.3333333333333333,
  0,
  0,
  0,
  0,
  0.3333333333333333,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [0, 0, 0, 0, 0.5, 0, 0, 0, 0, 0, 0, 0, 0, 0.5, 0, 0, 0, 0],
 [0, 0, 0, 0, 0.5, 0, 0, 0.5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0.3333333333333333,
  0,
  0,
  0,
  0,
  0.3333333333333333,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0.3333333333333333,
  0,
  0],
 [0, 0, 0, 0, 0.5, 0, 0, 0, 0, 0, 0, 0, 0, 0.5, 0, 0, 0, 0],
 [0, 0.5, 0, 0.5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0.3333333333333333,
  0.3333333333333333,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0.3333333333333333,
  0,
  0],
 [0,
  0,
  0,
  0,
  0.3333333333333333,
  0,
  0,
  0.33333333333333

In [14]:
dist_tensor = torch.tensor(genre_vector, device=dev, dtype=torch.float32)

In [15]:

item_tensor = torch.tensor(item2genre.item.tolist(), device=dev, dtype=torch.long)

In [16]:
item_tensor

tensor([   1,    2,    3,  ..., 3950, 3951, 3952], device='cuda:0')

In [17]:
dist_tensor.shape

torch.Size([3701, 18])

In [18]:


p_g_i = torch.zeros(size=(n_items, n_genres), dtype=torch.float32, device=dev)
p_g_i[item_tensor] = dist_tensor


In [20]:
item_tensor

tensor([   1,    2,    3,  ..., 3950, 3951, 3952], device='cuda:0')

In [21]:
p_g_i.shape

torch.Size([3953, 18])

In [22]:
teste = torch.tensor([1,2,0])
eps = 1e-9

In [24]:
torch.clamp(teste, eps)

tensor([1.0000e+00, 2.0000e+00, 1.0000e-09])